# Hamlet Next-Word Prediction with LSTM

This notebook demonstrates how a recurrent neural network can learn word patterns from a text corpus and predict the next word in a sequence.

The workflow is simple but important:
1. Read the text from Hamlet.
2. Tokenize the text into numeric word IDs.
3. Create overlapping word sequences so the model learns context.
4. Pad the sequences so every input has the same length.
5. Train an LSTM model to predict the next word.
6. Save the trained weights and reuse them for prediction.

This is a classic example of sequence modeling in natural language processing.

In [ ]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd

In [ ]:
data = gutenberg.raw('shakespeare-hamlet.txt')

with open('hamlet.txt', 'w') as file:
    file.write(data)

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [ ]:
with open('hamlet.txt', 'r') as file:
    text = file.read().lower()

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
total_words

In [ ]:
tokenizer.word_index

In [ ]:
# Create input sequences for the next-word prediction task.
# Each sentence is converted into overlapping n-grams, where each sequence is a prefix
# of the sentence and the model learns to predict the next word after that prefix.

input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i + 1]
        input_sequences.append(n_gram_sequence)

input_sequences

In [ ]:
# Determine the maximum sequence length across all training examples.
# This is important because all inputs to the LSTM must have the same length.
max_seq_len = max([len(x) for x in input_sequences])
max_seq_len

In [ ]:
# Pad all sequences to the same length so they can be used as a batch in the model.
# Padding is applied at the beginning of each sequence using the 'pre' option.
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))
input_sequences

In [ ]:
x,y = input_sequences[:, :-1], input_sequences[:, -1]

In [ ]:
# Convert target labels into one-hot encoded vectors for multi-class classification.
# Each target word is represented as a vector of length equal to the vocabulary size.
from tensorflow.keras.utils import to_categorical
y = to_categorical(y, num_classes=total_words)

In [ ]:
# Prepare training and testing data.
# We split the generated n-gram sequences into train and test sets so the model can learn
# patterns from one subset and be evaluated on unseen examples.
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [ ]:
# Build the language model architecture.
# The Embedding layer converts each integer token into a dense vector representation.
# The LSTM layers learn the sequence patterns in the text.
# The final Dense layer with a softmax activation predicts the probability of the next word.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Embedding
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential()

model.add(Embedding(input_dim=total_words, output_dim=100, input_length=max_seq_len))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
# Train the LSTM model.
# Early stopping is used to halt training if validation performance stops improving.
early_stop = EarlyStopping(patience=50, restore_best_weights=True)
history = model.fit(
    x_train, y_train,
    epochs=100,
    validation_data=(x_test, y_test),
    verbose=1,
    callbacks=[early_stop]
)

In [ ]:
# Load the trained weights from a previously saved model checkpoint.
# This is useful when training has already been completed and we want to reuse the model.
# The architecture must match exactly the one used during training.
model.load_weights('next_word_lstm.h5')

In [ ]:
def predict_next_word(model, tokenizer, text, max_seq_len):
    """Predict the next word for a given text prefix."""
    token_list = tokenizer.texts_to_sequences([text])[0]

    # Keep only the latest relevant tokens so the sequence length stays valid.
    if len(token_list) >= max_seq_len:
        token_list = token_list[-(max_seq_len - 1):]

    # Pad the sequence to the model's expected input length.
    token_list = pad_sequences([token_list], maxlen=max_seq_len, padding='pre')

    # Predict probabilities for all words in the vocabulary.
    predicted = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predicted[0])

    # Convert the predicted index back into a human-readable word.
    return tokenizer.index_word.get(predicted_index, None)

In [ ]:
# Example inference: pass a context string and ask the model for the next word.
input_text = 'like a Soldier to the'
print(f"Input Text: {input_text}")
next_word = predict_next_word(model, tokenizer, input_text, max_seq_len)
print(f"Next word: {next_word}")